# Early detection of diabetic retinopathy (grading only)
**Before running:** Settings -> Accelerator **GPU (T4 x2 or P100)**, Internet **On**. Add the three datasets
(DDR, EyePACS, APTOS 2019) and the dataset you made from `Early-DR.zip`. Read `RUN_INSTRUCTIONS.md` for timings.
Then run the cells in order.

In [ ]:
!pip install -q timm
import glob, os, shutil
settings_file = glob.glob("/kaggle/input/**/steps/settings.py", recursive=True)[0]   # the uploaded zip
shutil.copytree(os.path.dirname(os.path.dirname(settings_file)), "/kaggle/working/code", dirs_exist_ok=True)
%cd /kaggle/working/code
!python -c "import sys; sys.path.insert(0, 'steps'); import settings; print(settings.PATHS)"

The line above prints the dataset folders it found. If one is `None`, add that dataset or write its folder in `steps/settings.py`.

## Resume (only when continuing an earlier run)
Add the previous notebook's **output** as an input (Add Data -> Your Work / Notebook Output). This cell restores `splits.csv`, the processed images and all finished runs, so finished work is skipped. It does nothing on a first run.

In [ ]:
import glob, os, shutil
previous = [os.path.dirname(f) for f in glob.glob("/kaggle/input/**/splits.csv", recursive=True)]
if previous:
    print("Restoring from", previous[0])
    shutil.copytree(previous[0], "/kaggle/working", dirs_exist_ok=True, ignore=shutil.ignore_patterns("code"))
else:
    print("Nothing to restore: first run")

## 1. Prepare data (CPU work, ~20-60 min)
Counts per source/split.

In [ ]:
!python steps/01_prepare_data.py

## 2. Audit
Class distribution, merge table, image statistics, duplicates.

In [ ]:
!python steps/02_audit.py

## 3. Source check
Can the source be told from the image? (AUROC)

In [ ]:
!python steps/03_source_check.py

## 4. Train arms A, B, C
Validation scores. No test data is opened. Longest core step.

In [ ]:
!python steps/04_train_arms.py

## 5. Imbalance ablations
Pick the best sampler/loss/augmentation on validation, then edit `configs/imbalance/combined.yaml`. Skip if short of time.

In [ ]:
!python steps/05_imbalance_ablations.py

## Declare the analysis (before ANY test set is opened)
Read `DECLARED_ANALYSIS.md` (primary comparison: sensitivity for early DR at specificity 0.90). Running this cell stamps today's date on it, which unlocks steps 6, 8 and 10. Do it only after step 5, when you have settled your choices.

In [ ]:
!cat DECLARED_ANALYSIS.md
!sed -i "s/____/$(date +%F)/" DECLARED_ANALYSIS.md

## 6. Evaluate on internal test sets
Needs the declaration cell above. Primary result: sensitivity at the declared specificity.

In [ ]:
!python steps/06_evaluate_internal.py

## 7. Compare arms
Holm-corrected comparisons.

In [ ]:
!python steps/07_compare_arms.py

## 8. External test set (APTOS) - last
Only after model selection is finished.

In [ ]:
!python steps/08_evaluate_external.py

## 9. Saliency
Grad-CAM + randomised-weights sanity check.

In [ ]:
!python steps/09_saliency.py

## 10. Optional: stage 2 and extensions
Early vs late head, three-class, colour normalisation.

In [ ]:
!python steps/10_stage2_and_extensions.py

## 11. Figures
All plots.

In [ ]:
!python steps/11_figures.py

## 12. Are the results inflated by duplicates?
Run after steps 6 and 8. Uses the saved predictions only, so no test set is opened again. Prints, per arm and source, the specificity, grade-1 and grade-2 AUROC separately, and the metrics with and without images that have a duplicate in the training set. Look at `figures/duplicate_pairs.jpg` by eye.

In [ ]:
!python steps/12_check_duplicates.py
from IPython.display import Image, display
import os
if os.path.exists("/kaggle/working/figures/duplicate_pairs.jpg"):
    display(Image("/kaggle/working/figures/duplicate_pairs.jpg"))

## 13. Can simple image properties predict early DR on their own?
CPU only, a few minutes. Needs only steps 1-2. AUROC well above 0.5 means the label is entangled with image size, file size, colour or sharpness, so high CNN scores there may be an artefact.

In [ ]:
!python steps/13_shortcut_check.py